In [7]:
# Cell 1: Load reconstructed events + get first cart_time per session
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

events = pd.read_csv("../data/reconstructed_events_v3.csv")
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)
events = events.sort_values(["user_id", "event_time"]).reset_index(drop=True)

first_cart = (
    events[events["event_type"] == "cart"]
    .groupby("reconstructed_session_id")
    .agg(user_id=("user_id", "first"), cart_time=("event_time", "min"))
    .reset_index()
)

print("Total cart sessions:", first_cart.shape[0])
first_cart.head()

Total cart sessions: 42853


,reconstructed_session_id,user_id,cart_time
0,1515915625353286099_1,1515915625353286099,2020-10-03 11:23:00+00:00
1,1515915625353286099_2,1515915625353286099,2020-10-06 08:28:18+00:00
2,1515915625353457259_1,1515915625353457259,2020-09-29 05:51:58+00:00
3,1515915625353534622_2,1515915625353534622,2020-10-06 09:45:38+00:00
4,1515915625353561691_1,1515915625353561691,2020-11-24 17:35:50+00:00


In [4]:
# Cell 2: Build purchases table (user_id + purchase_time only, NO session_id)
purchases = events.loc[
    events["event_type"] == "purchase", ["user_id", "event_time"]
].rename(columns={"event_time": "purchase_time"})

# merge_asof requires the "on" column sorted globally by that column
purchases = purchases.sort_values("purchase_time").reset_index(drop=True)

print("Total purchase events:", purchases.shape[0])
purchases.head()

Total purchase events: 37343


,user_id,purchase_time
0,1515915625519390468,2020-09-24 12:04:10+00:00
1,1515915625513574486,2020-09-24 12:15:06+00:00
2,1515915625518130982,2020-09-24 12:19:01+00:00
3,1515915625518130982,2020-09-24 12:25:18+00:00
4,1515915625519356010,2020-09-24 12:29:49+00:00


In [5]:
# Cell 3: THE FIX -- match each cart to the user's NEXT purchase (any session)
ABANDONMENT_WINDOW_HOURS = 24  # we'll experiment with this value later

cart_lookup = first_cart.sort_values("cart_time").reset_index(drop=True)

next_purchase = pd.merge_asof(
    cart_lookup,
    purchases,
    left_on="cart_time",
    right_on="purchase_time",
    by="user_id",
    direction="forward",   # finds the next purchase AFTER cart_time, any session
)

next_purchase["hours_to_purchase"] = (
    next_purchase["purchase_time"] - next_purchase["cart_time"]
).dt.total_seconds() / 3600

# Abandoned = no purchase found at all, OR it happened outside the window
next_purchase["is_abandoned"] = (
    next_purchase["purchase_time"].isna()
    | (next_purchase["hours_to_purchase"] > ABANDONMENT_WINDOW_HOURS)
).astype(int)

print("Shape:", next_purchase.shape)
print(next_purchase["is_abandoned"].value_counts())
print(next_purchase["is_abandoned"].value_counts(normalize=True) * 100)
next_purchase.head()

Shape: (42853, 6)
is_abandoned
0    23304
1    19549
Name: count, dtype: int64
is_abandoned
0    54.381257
1    45.618743
Name: proportion, dtype: float64


,reconstructed_session_id,user_id,cart_time,purchase_time,hours_to_purchase,is_abandoned
0,1515915625519390468_1,1515915625519390468,2020-09-24 12:02:53+00:00,2020-09-24 12:04:10+00:00,0.021389,0
1,1515915625519392459_1,1515915625519392459,2020-09-24 12:08:24+00:00,NaT,NaN,1
2,1515915625513574486_1,1515915625513574486,2020-09-24 12:12:16+00:00,2020-09-24 12:15:06+00:00,0.047222,0
3,1515915625519356010_1,1515915625519356010,2020-09-24 12:13:03+00:00,2020-09-24 12:29:49+00:00,0.279444,0
4,1515915625517759569_1,1515915625517759569,2020-09-24 12:16:41+00:00,2020-09-28 12:59:30+00:00,96.713611,1


In [6]:
# Cell 4: Rebuild session-level history features (not event-level)
session_cart_flag = (
    events.groupby("reconstructed_session_id")
    .agg(
        user_id=("user_id", "first"),
        session_start=("event_time", "min"),
        session_has_cart=("event_type", lambda x: int((x == "cart").any())),
    )
    .reset_index()
    .sort_values(["user_id", "session_start"])
)

# previous_cart_session_count: how many EARLIER sessions had a cart-add (session-level, not event-level)
session_cart_flag["previous_cart_session_count"] = (
    session_cart_flag.groupby("user_id")["session_has_cart"]
    .cumsum()
    .shift(1)
    .fillna(0)
    .astype(int)
)

# session_has_purchase flag per session
session_purchase_flag = (
    events.groupby("reconstructed_session_id")["event_type"]
    .apply(lambda x: int((x == "purchase").any()))
    .reset_index(name="session_has_purchase")
)

session_history = session_cart_flag.merge(session_purchase_flag, on="reconstructed_session_id")

session_history["previous_purchase_count"] = (
    session_history.groupby("user_id")["session_has_purchase"]
    .cumsum()
    .shift(1)
    .fillna(0)
    .astype(int)
)
session_history["previous_session_count"] = session_history.groupby("user_id").cumcount()
session_history["has_previous_purchase"] = (session_history["previous_purchase_count"] > 0).astype(int)
session_history["user_purchase_rate"] = (
    session_history["previous_purchase_count"] / session_history["previous_session_count"].replace(0, np.nan)
).fillna(0)

print("Shape:", session_history.shape)
session_history[["previous_session_count", "previous_purchase_count", "previous_cart_session_count", "user_purchase_rate"]].describe()

Shape: (504630, 10)


,previous_session_count,previous_purchase_count,previous_cart_session_count,user_purchase_rate
count,504630.000000,504630.000000,504630.000000,504630.000000
mean,1.100666,0.094313,0.173840,0.017160
std,6.629243,0.369637,0.553474,0.112624
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000
max,265.000000,10.000000,15.000000,1.000000


In [5]:
# Cell 5: Behavioral features computed from events up to and including the cart event
events_with_cart_time = events.merge(
    first_cart[["reconstructed_session_id", "cart_time"]], on="reconstructed_session_id"
)
events_before_cart = events_with_cart_time[
    events_with_cart_time["event_time"] <= events_with_cart_time["cart_time"]
]

# Views before cart
views_before_cart = (
    events_before_cart[events_before_cart["event_type"] == "view"]
    .groupby("reconstructed_session_id")
    .size()
    .reset_index(name="views_before_cart")
)

# Price behavior
price_summary = (
    events_before_cart.groupby("reconstructed_session_id")
    .agg(
        avg_price_before_cart=("price", "mean"),
        min_price=("price", "min"),
        max_price=("price", "max"),
    )
    .reset_index()
)
price_summary["price_range_before_cart"] = price_summary["max_price"] - price_summary["min_price"]

# Brand / category diversity + total interactions
brand_cat_summary = (
    events_before_cart.groupby("reconstructed_session_id")
    .agg(
        unique_brands_before_cart=("brand", "nunique"),
        unique_categories_before_cart=("category_code", "nunique"),
        unique_products_before_cart=("product_id", "nunique"),
        total_interactions_before_cart=("event_type", "size"),
    )
    .reset_index()
)

# Duration before cart
duration_summary = (
    events_before_cart.groupby("reconstructed_session_id")
    .agg(session_start=("event_time", "min"))
    .reset_index()
    .merge(first_cart[["reconstructed_session_id", "cart_time"]], on="reconstructed_session_id")
)
duration_summary["duration_before_cart"] = (
    duration_summary["cart_time"] - duration_summary["session_start"]
).dt.total_seconds() / 60

print("views_before_cart shape:", views_before_cart.shape)
print("price_summary shape:", price_summary.shape)
print("brand_cat_summary shape:", brand_cat_summary.shape)
print("duration_summary shape:", duration_summary.shape)

price_summary.describe()

views_before_cart shape: (42006, 2)
price_summary shape: (42853, 5)
brand_cat_summary shape: (42853, 5)
duration_summary shape: (42853, 4)


,avg_price_before_cart,min_price,max_price,price_range_before_cart
count,42853.000000,42853.000000,42853.000000,42853.000000
mean,158.006122,149.620400,168.772383,19.151983
std,190.952065,183.938132,293.402383,223.794365
min,0.370000,0.370000,0.370000,0.000000
25%,29.220000,27.620000,30.100000,0.000000
50%,80.266667,74.210000,82.970000,0.000000
75%,226.840000,214.100000,239.970000,0.000000
max,4071.730000,4071.730000,42590.130000,42496.500000


In [6]:
# Cell 6: days_since_last_purchase -- last purchase strictly BEFORE this cart,
# across the user's full history (any session), using backward merge_asof
prev_purchase = pd.merge_asof(
    cart_lookup,
    purchases,
    left_on="cart_time",
    right_on="purchase_time",
    by="user_id",
    direction="backward",
    allow_exact_matches=False,
)

prev_purchase["days_since_last_purchase"] = (
    (prev_purchase["cart_time"] - prev_purchase["purchase_time"]).dt.total_seconds() / 86400
).fillna(-1)

print("Shape:", prev_purchase.shape)
prev_purchase["days_since_last_purchase"].describe()

Shape: (42853, 5)


count    42853.000000
mean        -0.449238
std          3.970760
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         -1.000000
max        143.037465
Name: days_since_last_purchase, dtype: float64

In [7]:
# Cell 7: Assemble the final model-ready dataset
model_df = first_cart[["reconstructed_session_id", "cart_time"]].copy()

# Target
model_df = model_df.merge(
    next_purchase[["reconstructed_session_id", "is_abandoned"]],
    on="reconstructed_session_id"
)

# Session-level history features (filter to cart sessions only)
model_df = model_df.merge(
    session_history[[
        "reconstructed_session_id",
        "previous_session_count",
        "previous_purchase_count",
        "has_previous_purchase",
        "user_purchase_rate",
        "previous_cart_session_count",
    ]],
    on="reconstructed_session_id"
)

# Behavioral features
model_df = model_df.merge(views_before_cart, on="reconstructed_session_id", how="left")
model_df = model_df.merge(
    price_summary[["reconstructed_session_id", "avg_price_before_cart", "price_range_before_cart"]],
    on="reconstructed_session_id", how="left"
)
model_df = model_df.merge(brand_cat_summary, on="reconstructed_session_id", how="left")
model_df = model_df.merge(
    duration_summary[["reconstructed_session_id", "duration_before_cart"]],
    on="reconstructed_session_id", how="left"
)
model_df = model_df.merge(
    prev_purchase[["reconstructed_session_id", "days_since_last_purchase"]],
    on="reconstructed_session_id", how="left"
)

# Fill missing views with 0, then derive ratio features
model_df["views_before_cart"] = model_df["views_before_cart"].fillna(0).astype(int)
model_df["cart_to_view_ratio"] = 1 / model_df["views_before_cart"].replace(0, 1)
model_df["no_view_before_cart"] = (model_df["views_before_cart"] == 0).astype(int)

# Time-based features
model_df["cart_hour"] = model_df["cart_time"].dt.hour
model_df["cart_day_of_week"] = model_df["cart_time"].dt.dayofweek
model_df["cart_is_weekend"] = (model_df["cart_day_of_week"] >= 5).astype(int)

model_df = model_df.drop(columns=["cart_time"]).fillna(0)

print("Final shape:", model_df.shape)
print("Missing values:", model_df.isnull().sum().sum())
print("Duplicate sessions:", model_df["reconstructed_session_id"].duplicated().sum())
model_df["is_abandoned"].value_counts()

Final shape: (42853, 21)
Missing values: 0
Duplicate sessions: 0


is_abandoned
0    23304
1    19549
Name: count, dtype: int64

In [8]:
# Cell 8: Sanity check -- correlation with the NEW, corrected target
correlations = model_df.corr(numeric_only=True)["is_abandoned"].sort_values()
print(correlations)

user_purchase_rate               -0.127910
has_previous_purchase            -0.115489
previous_purchase_count          -0.108566
days_since_last_purchase         -0.064927
previous_cart_session_count      -0.059479
duration_before_cart             -0.038085
previous_session_count           -0.028762
no_view_before_cart              -0.014605
total_interactions_before_cart   -0.006174
views_before_cart                -0.004187
unique_products_before_cart       0.004189
cart_to_view_ratio                0.004443
price_range_before_cart           0.007585
cart_day_of_week                  0.011422
cart_is_weekend                   0.011822
unique_brands_before_cart         0.016891
unique_categories_before_cart     0.019594
cart_hour                         0.027207
avg_price_before_cart             0.087188
is_abandoned                      1.000000
Name: is_abandoned, dtype: float64


In [9]:
# Cell 9: Train and evaluate the model on the corrected target
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

X = model_df.drop(columns=["reconstructed_session_id", "is_abandoned"])
y = model_df["is_abandoned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)
print("Baseline accuracy:", accuracy_score(y_test, baseline.predict(X_test)))

model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))
print("ROC-AUC  :", roc_auc_score(y_test, proba))
print(confusion_matrix(y_test, pred))

Baseline accuracy: 0.5438105238595263
Accuracy : 0.5837125189592813
Precision: 0.5575757575757576
Recall   : 0.4235294117647059
F1 Score : 0.4813953488372093
ROC-AUC  : 0.6080476512125703
[[3347 1314]
 [2254 1656]]


In [11]:
# Cell 10: Attach the exact product/category of the cart-added item, then build
# a LEAK-FREE historical abandonment rate per category (uses only sessions that
# happened strictly BEFORE the current one, in time order).

# Get the exact product_id / category_code / brand of the item added to cart
first_cart_events = (
    events[events["event_type"] == "cart"]
    .sort_values("event_time")
    .drop_duplicates(subset="reconstructed_session_id", keep="first")
    [["reconstructed_session_id", "product_id", "category_code", "brand"]]
)

cart_context = first_cart.merge(first_cart_events, on="reconstructed_session_id", how="left")
cart_context = cart_context.merge(
    next_purchase[["reconstructed_session_id", "is_abandoned"]], on="reconstructed_session_id"
)
cart_context["category_code"] = cart_context["category_code"].fillna("unknown")
cart_context = cart_context.sort_values("cart_time").reset_index(drop=True)

# --- Leak-free category historical abandonment rate ---
# Shift the target by 1 within each category (in time order), then take a
# running sum/count. This means: for each session, we only use the OUTCOMES
# of earlier sessions in the same category -- never the current session's
# own label, and never anything from the future.
cart_context["is_abandoned_shifted"] = cart_context.groupby("category_code")["is_abandoned"].shift(1)
cart_context["cat_prior_abandon_sum"] = (
    cart_context.groupby("category_code")["is_abandoned_shifted"].cumsum()
)
cart_context["cat_prior_count"] = cart_context.groupby("category_code").cumcount()

overall_abandon_rate = cart_context["is_abandoned"].mean()  # fallback for brand-new categories
cart_context["category_historical_abandon_rate"] = (
    cart_context["cat_prior_abandon_sum"] / cart_context["cat_prior_count"].replace(0, np.nan)
).fillna(overall_abandon_rate)

# --- Product popularity: how many times has this EXACT product been cart-added before ---
cart_context["product_prior_cart_count"] = cart_context.groupby("product_id").cumcount()

print("Shape:", cart_context.shape)
print("Overall abandon rate (fallback value):", overall_abandon_rate)
cart_context[["category_historical_abandon_rate", "product_prior_cart_count"]].describe()

Shape: (42853, 12)
Overall abandon rate (fallback value): 0.456187431451707


,category_historical_abandon_rate,product_prior_cart_count
count,42853.000000,42853.000000
mean,0.457371,58.948988
std,0.082131,136.012912
min,0.000000,0.000000
25%,0.422675,1.000000
50%,0.441604,8.000000
75%,0.516242,44.000000
max,1.000000,1089.000000


In [12]:
# Cell 11: Within-session behavior specific to the EXACT product that was cart-added

cart_item_info = cart_context[["reconstructed_session_id", "product_id", "category_code"]].rename(
    columns={"product_id": "cart_product_id", "category_code": "cart_category_code"}
)

# events_before_cart was already built in Cell 5
events_before_cart_ctx = events_before_cart.merge(cart_item_info, on="reconstructed_session_id")

# Views of the EXACT product that ended up in the cart, before the cart-add
same_product_views = events_before_cart_ctx[
    (events_before_cart_ctx["event_type"] == "view")
    & (events_before_cart_ctx["product_id"] == events_before_cart_ctx["cart_product_id"])
]

product_repeat_views = (
    same_product_views.groupby("reconstructed_session_id").size()
    .reset_index(name="product_repeat_views_before_cart")
)

# Minutes between the FIRST view of that exact product and the cart-add moment
first_view_of_cart_product = (
    same_product_views.groupby("reconstructed_session_id")["event_time"].min()
    .reset_index(name="first_view_time")
)
view_to_cart = first_view_of_cart_product.merge(
    first_cart[["reconstructed_session_id", "cart_time"]], on="reconstructed_session_id"
)
view_to_cart["view_to_cart_minutes"] = (
    view_to_cart["cart_time"] - view_to_cart["first_view_time"]
).dt.total_seconds() / 60

# Browsing focus: fraction of pre-cart views that were in the SAME category as the cart item
views_only_ctx = events_before_cart_ctx[events_before_cart_ctx["event_type"] == "view"]
category_focus = (
    views_only_ctx.assign(same_category=lambda d: d["category_code"] == d["cart_category_code"])
    .groupby("reconstructed_session_id")["same_category"].mean()
    .reset_index(name="category_view_concentration")
)

print("product_repeat_views shape:", product_repeat_views.shape)
print("view_to_cart shape:", view_to_cart.shape)
print("category_focus shape:", category_focus.shape)
category_focus.describe()

product_repeat_views shape: (41937, 2)
view_to_cart shape: (41937, 4)
category_focus shape: (42006, 2)


,category_view_concentration
count,42006.000000
mean,0.987685
std,0.082512
min,0.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,1.000000


In [13]:
# Cell 12: Merge new features into model_df, then retrain and re-evaluate

model_df_v2 = model_df.merge(
    cart_context[["reconstructed_session_id", "category_historical_abandon_rate", "product_prior_cart_count"]],
    on="reconstructed_session_id", how="left"
)
model_df_v2 = model_df_v2.merge(
    product_repeat_views, on="reconstructed_session_id", how="left"
)
model_df_v2 = model_df_v2.merge(
    view_to_cart[["reconstructed_session_id", "view_to_cart_minutes"]],
    on="reconstructed_session_id", how="left"
)
model_df_v2 = model_df_v2.merge(
    category_focus, on="reconstructed_session_id", how="left"
)

# Missing product_repeat_views_before_cart means the user never re-viewed the
# exact cart item before adding it -- fill with 0 and add an explicit flag
model_df_v2["never_viewed_cart_item_before"] = model_df_v2["product_repeat_views_before_cart"].isna().astype(int)
model_df_v2["product_repeat_views_before_cart"] = model_df_v2["product_repeat_views_before_cart"].fillna(0)

# Missing view_to_cart_minutes means no prior view of that exact item existed
model_df_v2["view_to_cart_minutes"] = model_df_v2["view_to_cart_minutes"].fillna(-1)

# Missing category_view_concentration means no views at all before cart (already flagged as no_view_before_cart)
model_df_v2["category_view_concentration"] = model_df_v2["category_view_concentration"].fillna(0)

print("New shape:", model_df_v2.shape)
print("Missing values:", model_df_v2.isnull().sum().sum())

# Correlation check with the new features included
print(model_df_v2.corr(numeric_only=True)["is_abandoned"].sort_values())

New shape: (42853, 27)
Missing values: 0
user_purchase_rate                 -0.127910
has_previous_purchase              -0.115489
previous_purchase_count            -0.108566
days_since_last_purchase           -0.064927
previous_cart_session_count        -0.059479
view_to_cart_minutes               -0.045656
duration_before_cart               -0.038085
previous_session_count             -0.028762
never_viewed_cart_item_before      -0.017774
product_repeat_views_before_cart   -0.016798
no_view_before_cart                -0.014605
total_interactions_before_cart     -0.006174
views_before_cart                  -0.004187
category_view_concentration        -0.000132
unique_products_before_cart         0.004189
cart_to_view_ratio                  0.004443
price_range_before_cart             0.007585
cart_day_of_week                    0.011422
cart_is_weekend                     0.011822
unique_brands_before_cart           0.016891
unique_categories_before_cart       0.019594
cart_hour     

In [3]:
# Cell 13: Retrain and evaluate with the new features
X2 = model_df_v2.drop(columns=["reconstructed_session_id", "is_abandoned"])
y2 = model_df_v2["is_abandoned"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=10, stratify=y2
)

model_v2 = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
model_v2.fit(X2_train, y2_train)
pred2 = model_v2.predict(X2_test)
proba2 = model_v2.predict_proba(X2_test)[:, 1]

print("Accuracy :", accuracy_score(y2_test, pred2))
print("Precision:", precision_score(y2_test, pred2))
print("Recall   :", recall_score(y2_test, pred2))
print("F1 Score :", f1_score(y2_test, pred2))
print("ROC-AUC  :", roc_auc_score(y2_test, proba2))
print(confusion_matrix(y2_test, pred2))

# Feature importance -- which features is the model actually leaning on?
importances = pd.Series(model_v2.feature_importances_, index=X2.columns).sort_values(ascending=False)
print("\nTop 10 feature importances:")
print(importances.head(20))

NameError: name 'model_df_v2' is not defined

In [20]:
# Cell 14: Hyperparameter tuning via RandomizedSearchCV (5-fold CV)
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_distributions = {
    "n_estimators": randint(100, 400),
    "learning_rate": uniform(0.01, 0.15),
    "max_depth": randint(2, 6),
    "min_samples_leaf": randint(5, 50),
    "subsample": uniform(0.6, 0.4),
}

base_model = GradientBoostingClassifier(random_state=20)

search = RandomizedSearchCV(
    base_model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="roc_auc",
    cv=5,
    random_state=20,
    n_jobs=-1,
    verbose=1,
)

search.fit(X2_train, y2_train)

print("Best parameters:", search.best_params_)
print("Best CV ROC-AUC:", search.best_score_)

# Evaluate the tuned model on the held-out test set
tuned_model = search.best_estimator_
tuned_pred = tuned_model.predict(X2_test)
tuned_proba = tuned_model.predict_proba(X2_test)[:, 1]

print("\n--- Tuned model: test set results ---")
print("Accuracy :", accuracy_score(y2_test, tuned_pred))
print("Precision:", precision_score(y2_test, tuned_pred))
print("Recall   :", recall_score(y2_test, tuned_pred))
print("F1 Score :", f1_score(y2_test, tuned_pred))
print("ROC-AUC  :", roc_auc_score(y2_test, tuned_proba))
print(confusion_matrix(y2_test, tuned_pred))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'learning_rate': np.float64(0.015383437842527995), 'max_depth': 5, 'min_samples_leaf': 39, 'n_estimators': 218, 'subsample': np.float64(0.8074043781792323)}
Best CV ROC-AUC: 0.6204743299846636

--- Tuned model: test set results ---
Accuracy : 0.5947270181987867
Precision: 0.5665445665445665
Recall   : 0.47468030690537083
F1 Score : 0.5165599777344837
ROC-AUC  : 0.6297758116172438
[[1621  710]
 [1027  928]]


In [8]:
# model_v2, X2_test, y2_test should already exist from Cell 12/13
explainer = shap.TreeExplainer(model_v2)

# Use a sample of the test set for speed (2000 is plenty for stable results)
sample_size = min(2000, X2_test.shape[0])
X_sample = X2_test.sample(n=sample_size, random_state=42)

shap_values = explainer.shap_values(X_sample)

# Global feature importance: mean absolute SHAP value per feature
global_importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=X_sample.columns
).sort_values(ascending=False)

print("Global SHAP feature importance:")
print(global_importance)

# Summary bar plot
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("shap_summary_bar.png", dpi=150)
plt.show()

NameError: name 'model_v2' is not defined